# Message Passing Neural Networks (MPNN)
## Spectral vs. Spatial GNNs

Graph Neural Networks can be approached from two complementary perspectives:

1. **Spectral perspective** – defines convolution via the graph Fourier transform and Laplacian eigen-decomposition.  
2. **Spatial perspective** – defines convolution via message passing and local neighborhood aggregation.

Modern GNNs are mostly spatial, but spectral theory provides the underlying rationale.



### Spectral GNNs

We got familiar with spectral GNN.

**Properties:**

- Requires eigen-decomposition → costly $O(n^3)$  
- Filters depend on the graph → limited transferability  
- Provides a theoretical foundation for low-pass filtering and smoothing  

**Examples:** ChebNet, Spectral CNN, GCN (first-order approximation)



### Spatial GNNs

Convolution directly on nodes via **message passing**:

$h_i^{(k+1)} = \phi^{(k)}\Big(h_i^{(k)}, Agg_{j \in \mathcal{N}(i)} Ψ ^{(k)}(h_i^{(k)}, h_j^{(k)}, e_{ij})\Big)$



- $h_i^{(k)}$: node feature at layer $k$  
- $Ψ$: message function  
- $Agg$: aggregation (sum, mean, max)  
- $\phi$: update function  
- $e_{ij}$: optional edge features  

**Advantages:**

- No eigenvectors, scales to large graphs  
- Transferable to unseen graphs  
- Intuitive: each node updates by looking at neighbors  

**Examples:** GraphSAGE, GAT, GIN, MPNNs, Graph Transformers



### Key Insights

| Aspect | Spectral | Spatial |
|--------|----------|---------|
| Foundation | Fourier & Laplacian | Message passing |
| Filter | Frequency domain | Node domain |
| Transferability | Limited | High |
| Scalability | Hard for large graphs | Easy |
| Examples | ChebNet, GCN | GAT, GraphSAGE |

**Unification:**  

- Spatial aggregation = low-pass filtering  
- Attention = adaptive spectral filter  
- Diffusion-based methods = heat-kernel spectral filtering  
- Polynomial filters = localized spectral operators  

Spectral theory gives the **language**, spatial models provide the **practical implementation**.

---

## Message Passing Neural Networks (MPNN): 
#### The Unifying Framework for Spatial GNNs

After understanding spectral graph convolution and Laplacian-based filtering, we now move to the **spatial** viewpoint of GNNs, where information flows locally along edges.  
Most modern GNNs — including GCN, GraphSAGE, GAT, and GIN — can be described under a single general model called the **Message Passing Neural Network (MPNN) framework**.

This formulation captures the essence of how nodes exchange information in a graph.

---

## The General MPNN Layer

An MPNN layer updates node representations by exchanging *messages* between neighbors.

For each node $ i $, the update rule is:

$$
h_i^{(l+1)} 
= 
\phi\!\left(
h_i^{(l)},\;
\text{AGG}\_{\;j \in N(i)}
\; \psi\!\left(h_i^{(l)},\, h_j^{(l)},\, e_{ij}\right)
\right).
$$

Here:

- $ h_i^{(l)} $: embedding of node $i$ at layer $l$  
- $ e_{ij} $: optional edge features  
- $ \psi $: **message function**  
- $ \text{AGG} $: a permutation-invariant **aggregation**  
- $ \phi $: **update function**  

The layer consists of three conceptual steps:

---

### Step 1 — Message Function

Each neighbor $j$ sends a message to $i$:

$$
m_{ij}^{(l)} = \psi(h_i^{(l)},\, h_j^{(l)},\, e_{ij}).
$$

This function can be:

- a linear transformation,  
- an MLP,  
- attention-based weights (as in GAT),  
- or any learnable function.

The message may depend on:

- sender node $j$,  
- receiver node $i$,  
- and edge features $e_{ij}$.

---

### Step 2 — Aggregation Function

Node $i$ collects messages from all neighbors:

$$
m_i^{(l)} = \text{AGG}_{j \in N(i)} \; m_{ij}^{(l)}.
$$

Aggregation must be **permutation invariant**, since graph neighborhoods have no order.

Common choices:

- **Sum** → used in GIN (most expressive)  
- **Mean** → used in GraphSAGE  
- **Max** → captures strongest signal  
- **Attention-weighted sum** → used in GAT  

---

### Step 3 — Update Function

The node updates its embedding:

$$
h_i^{(l+1)} = \phi\!\left(h_i^{(l)},\; m_i^{(l)}\right).
$$

Typical update functions:

- concatenation + MLP (GraphSAGE)  
- GRU/LSTM-style gating (Gated GNNs)  
- simple linear transform (GCN style)  
- residual connections  

---

## How Popular GNN Models Fit Into the MPNN Framework

### **GCN (Kipf & Welling, 2017)**  
A *linearized* MPNN with:

- message: $ \psi(h_j) = W h_j $
- aggregation: **normalized sum** using $D^{-1/2} A D^{-1/2}$
- update: no dependence on $h_i$

GCN message passing:

$$
h_i^{(l+1)} = \sigma \!\left( \sum_{j \in N(i) \cup \{i\}} \frac{1}{\sqrt{d_i d_j}} \, W h_j^{(l)} \right).
$$

---

### **GraphSAGE (Hamilton et al., 2017)**  
An inductive MPNN with flexible aggregators:

- sum/mean/max aggregators  
- update: concatenation with self-embedding  

$$
h_i^{(l+1)} = \sigma\!\left( W \cdot \left[ h_i^{(l)} \;\Vert\; \text{AGG}_{j \in N(i)} h_j^{(l)} \right] \right).
$$

---

### **GAT (Graph Attention Networks)**  
A message function computed through **attention scores**:

$$
\alpha_{ij}
=
\text{softmax}_j 
\big(
a^\top [W h_i \;\Vert\; W h_j]
\big).
$$

Message passing becomes:

$$
h_i^{(l+1)} = \sigma\!\left( \sum_{j \in N(i)} \alpha_{ij} W h_j \right).
$$

---

### **GIN (Graph Isomorphism Network)**  
Uses **sum** aggregation for maximal expressive power:

$$
h_i^{(l+1)} = \text{MLP} \left( (1 + \epsilon) h_i^{(l)} + \sum_{j \in N(i)} h_j^{(l)} \right).
$$

It matches the power of the Weisfeiler–Leman test.

---

## Spectral vs. Message Passing: Two Views of the Same Idea

| Aspect | Spectral View | Spatial / MPNN View |
|-------|---------------|---------------------|
| Based on | Graph Laplacian, eigenvectors | Local neighborhoods, message passing |
| Convolution | Defined via filtering in frequency domain | Defined via aggregating neighbor features |
| Computation | Global (needs whole graph) | Local (scalable) |
| Examples | ChebNet, GCN | GraphSAGE, GAT, GIN |
| Intuition | Heat diffusion, smoothing | Information exchange along edges |

**Both views describe the same underlying phenomenon:**  
**information mixing across connected nodes.**

Spectral = mathematical foundation  
MPNN = general practical framework

---

## Why the MPNN Framework Matters

This unified view helps us understand:

- how seemingly different GNNs are related,  
- how to design new architectures,  
- how message functions and aggregators determine expressiveness,  
- how edge features and directionality can be incorporated,  
- how GNNs can scale to large graphs.

It is the conceptual bridge between classical spectral theory and modern deep GNN design.

---

## Summary

- MPNN is a general framework that unifies most spatial GNN architectures.
- Each layer consists of message computation, aggregation, and update.
- GCN, GraphSAGE, GAT, and GIN are all special cases of MPNN.
- Spectral and spatial perspectives are different interpretations of the same process.



# Expressiveness of GNNs and the Weisfeiler–Leman Test

Graph Neural Networks (GNNs) are powerful, but not all GNN architectures can distinguish every pair of non-isomorphic graphs. The most common theoretical tool for reasoning about the distinguishing power of (message-passing) GNNs is the **Weisfeiler–Leman (WL) color refinement test** — usually the 1-dimensional version, called **1-WL** or **color refinement**.

## 1. The 1-WL (Color Refinement) Algorithm — Intuition and Steps

1-WL is an iterative vertex-coloring procedure. Initially, every node is assigned a color that encodes its input label (or a single uniform color if there are no input labels). At each round, every node updates its color by hashing (or otherwise compressing) the multiset formed by its current color together with the multiset of neighbor colors.

Formally, if $c^{(t)}(v)$ is the color of node $v$ at iteration $t$, then

$$
c^{(t+1)}(v) \leftarrow \text{hash}\Big( c^{(t)}(v),\ \{ c^{(t)}(u) : u \in \mathcal{N}(v) \} \Big).
$$

- The algorithm stops when the coloring stabilizes (no new colors appear).
- If the final color histograms (counts of each color) differ between two graphs, 1-WL certifies they are non-isomorphic.
- If 1-WL produces the same color histogram on two graphs, the test is **inconclusive** — the graphs may still be non-isomorphic.

### Key properties

- **Fast**: each iteration is local and typically linear in edges.
- **Incomplete**: there exist non-isomorphic graphs that 1-WL cannot distinguish.
- **Hierarchy**: higher-dimensional WL tests (k-WL for $k>1$) are strictly more powerful.

## 2. Relation between 1-WL and Message-Passing GNNs

A standard result in theory of GNNs:

- **Any standard Message-Passing Neural Network (MPNN)** — where node features are iteratively updated using an aggregation of neighbor features followed by an update function — **cannot be more powerful than 1-WL** in distinguishing non-isomorphic graphs (provided the aggregation and update are permutation invariant and the network uses fixed-width continuous features). Intuition: each GNN layer performs a continuous analogue of the WL color refinement step (aggregate neighbor information → combine with node’s own feature).

- **Graph Isomorphism Network (GIN)** (Xu et al., 2019) provably matches the discriminative power of 1-WL under suitable choices of aggregation and injective combine functions. GIN uses sum aggregation and multi-layer perceptrons (MLPs) to make the update approximately injective:

  $$
  h_v^{(k+1)} = \text{MLP}^{(k)}\Big( (1+\epsilon^{(k)})\cdot h_v^{(k)} + \sum_{u \in \mathcal{N}(v)} h_u^{(k)} \Big).
  $$

  With injective MLPs and appropriate parameters, GIN is as powerful as 1-WL.

## 3. Consequences for GNN Design

- If your task requires distinguishing graph structures that 1-WL cannot separate, standard MPNNs will also fail. To gain more expressiveness you can:
  - Use higher-order GNNs (related to k-WL) that operate on tuples of nodes.
  - Augment node features with additional structural information (positional encodings, random features, subgraph IDs).
  - Use architectures that incorporate subgraph or path information (subgraph GNNs, higher-order message passing).
- However, more expressive models often cost more computation and memory, so there is a trade-off.

#### Simple Python demonstration: 1-WL color refinement

The snippet below demonstrates one round of 1-WL color refinement on a small graph. 



In [ ]:
import networkx as nx
from collections import defaultdict
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

def weisfeiler_lehman_step(G, labels):
    """Perform one WL refinement step."""
    new_labels = {}
    for node in G.nodes():
        # Multiset of neighbors' labels
        neighbor_labels = [labels[neigh] for neigh in G.neighbors(node)]
        # Sort and convert to tuple for hashable key
        neighbor_labels_sorted = tuple(sorted(neighbor_labels))
        # New label is combination of current label and neighbors
        new_labels[node] = hash((labels[node], neighbor_labels_sorted))
    return new_labels

def wl_test(G1, G2, num_iter=5):
    """Perform WL test to check if two graphs are distinguishable."""
    # Initial labels: all 0
    labels1 = {n: 0 for n in G1.nodes()}
    labels2 = {n: 0 for n in G2.nodes()}

    for i in range(num_iter):
        labels1 = weisfeiler_lehman_step(G1, labels1)
        labels2 = weisfeiler_lehman_step(G2, labels2)
        
        # Compare multisets of labels
        multiset1 = sorted(labels1.values())
        multiset2 = sorted(labels2.values())
        print(f"Step {i+1} - Graph A labels: {multiset1}")
        print(f"Step {i+1} - Graph B labels: {multiset2}\n")
        
        if multiset1 != multiset2:
            return True  # Graphs are distinguishable
    return False  # Graphs are indistinguishable

# --- Example graphs ---
# Graph A: 4-node cycle
G_A = nx.cycle_graph(4)
# Graph B: 4-node cycle + diagonal
G_B = nx.cycle_graph(4)
G_B.add_edge(0, 2)

# --- Run WL test ---
distinguishable = wl_test(G_A, G_B)
print("Graphs are distinguishable by 1-WL:" , distinguishable)


Step 1 - Graph A labels: [7676811509268022211, 7676811509268022211, 7676811509268022211, 7676811509268022211]
Step 1 - Graph B labels: [3222727695683937112, 3222727695683937112, 7676811509268022211, 7676811509268022211]

Graphs are distinguishable by 1-WL: True


In [9]:
# --- Simple 2-layer GCN ---
class SimpleGCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # First GCN layer: input_dim=1, hidden_dim=2
        self.conv1 = GCNConv(1, 2)
        # Second GCN layer: hidden_dim=2, output_dim=2
        self.conv2 = GCNConv(2, 2)

    def forward(self, x, edge_index):
        # First layer + ReLU activation
        x = self.conv1(x, edge_index).relu()
        # Second layer (no activation)
        x = self.conv2(x, edge_index)
        return x

# Instantiate the model
model = SimpleGCN()

# --- Compute embeddings for both graphs ---
out_A = model(graph_A.x, graph_A.edge_index)
out_B = model(graph_B.x, graph_B.edge_index)

# Print the embeddings
print("Graph A embeddings:\n", out_A.detach())
print("Graph B embeddings:\n", out_B.detach())

Graph A embeddings:
 tensor([[ 1.2406, -0.9831],
        [ 1.2406, -0.9831],
        [ 1.2406, -0.9831],
        [ 1.2406, -0.9831]])
Graph B embeddings:
 tensor([[ 1.3205, -1.0464],
        [ 1.1482, -0.9099],
        [ 1.3205, -1.0464],
        [ 1.1482, -0.9099]])


### Analysis of examples

- **Small differences ≠ identical embeddings**  
  Although the 1-WL test can perfectly distinguish Graph A and B, the GCN embeddings are still close. In downstream tasks like classification, such small differences might lead to errors.

- **Reasons for closeness:**  
  1. All nodes have identical initial features → GCN only aggregates neighbor information.  
  2. Few layers (2-layer GCN) → global structural differences are not fully captured.  
  3. Aggregation via mean/sum → a single extra edge has limited impact on embeddings.

### Summary

- The embeddings are not identical, but a simple GCN has **limited discriminative power**.  
- To increase separation and obtain clearer embeddings:  
  - Use GIN or a more expressive GNN.  
  - Add node features (e.g., degree or random features).  
  - Increase the number of layers.


This simple routine shows how node colors are refined. Repeating wl_one_round until convergence implements the full 1-WL procedure.

Takeaway: 1-WL is the canonical baseline for theoretical expressiveness of GNNs. Standard MPNNs are bounded by 1-WL; GIN reaches that bound. If you need strictly greater discriminative power, consider higher-order or augmented architectures.



# Oversquashing

**Bottlenecks and Graph Curvature**

In graph learning, a key limitation is **oversquashing**: the phenomenon that prevents distant information from being transmitted reliably through a GNN.  
Oversquashing is **different from oversmoothing**; both are important but distinct failure modes.

---

## 1. What is Oversquashing?

**Oversquashing** occurs when a large amount of information (many messages) must pass through a **small number of edges or nodes** — a bottleneck. Because typical message-passing layers compress neighbor information into a fixed-size vector, the node(s) at the bottleneck cannot faithfully encode all upstream information.

Consequences:

- Long-range dependencies are "squashed" into low-dimensional representations.  
- Performance degrades for tasks that require combining information from distant parts of the graph (e.g., parity-like tasks, long-range relational reasoning).

**Formal scenario:**  
Consider regions \(A\) and \(B\) in a graph. If all paths from \(A\) to \(B\) go through a small cut \(C\) (few edges/nodes), the total information from \(A\) must be compressed through \(C\), causing loss.

---

## 2. Intuition via Message Passing

A typical MPNN layer aggregates neighbor messages:

\[
m_v^{(k)} = \text{AGG}\big( \{ h_u^{(k)} : u \in \mathcal{N}(v) \}\big), \qquad
h_v^{(k+1)} = \phi( h_v^{(k)}, m_v^{(k)} ).
\]

If many messages are aggregated at a bottleneck node \(b\), the resulting fixed-length vector **cannot uniquely encode all combinations of upstream messages**.

---

## 3. A Simple Illustrative Example

- Construct two clusters \(A\) and \(B\), each with many nodes, connected through a **single bridge node** \(b\).  
- Nodes in \(A\) have signals that must influence a node in \(B\).  
- All information must pass through \(b\), which aggregates multiple inputs into one vector.  

**Consequence:** Different configurations of \(A\) collapse to the same representation at \(b\). The network cannot distinguish them.

### Small Numerical Demonstration (Conceptual)

- Let cluster \(A\) have 100 nodes, each carrying one bit.  
- The bridge node can only pass a 32-dimensional float vector.  
- The number of distinct messages is limited (\(2^{32}\)) compared to \(2^{100}\) possible input configurations.  

> This illustrates the information bottleneck: many-to-one mapping through narrow channels.

---

## 4. Relation to Graph Curvature and Bottlenecks

Research links oversquashing to geometric/topological properties:

- **Negative curvature** and **low-conductance cuts** → strong bottlenecks → increased oversquashing.  
- Tree-like branches and narrow bridges compress information more than well-connected meshes.

**Graph curvature intuition:** Measures neighborhood expansion; narrow cuts or low expansion indicate potential oversquashing.

---

## 5. Connection to Diffusion and Heat Interpretation

- Message passing is analogous to **local diffusion / smoothing**.  
- Propagating detailed information over long geodesic distances attenuates and mixes signals.  
- Bottlenecks accelerate this loss: important high-frequency or combinatorial information may vanish before reaching the target.

**Comparison with oversmoothing:**

- **Oversmoothing:** Node representations converge to similar vectors due to repeated smoothing (large effective \(t\) in diffusion).  
- **Oversquashing:** Node representations do not converge, but bottlenecks limit the **capacity** to transmit diverse information.



## 6. Remedies and Design Patterns to Mitigate Oversquashing

Several strategies can help reduce oversquashing:

- **Increase channel width**  
  Larger hidden dimensions can carry more information. This is helpful but not a complete solution and increases the number of parameters.

- **Add skip/residual connections or residual gating**  
  Allow earlier (less-compressed) signals to bypass bottlenecks, preserving more information.

- **Use virtual nodes**  
  Add fully-connected virtual nodes that gather global information and broadcast it back to the graph.

- **Graph rewiring / edge augmentation**  
  Add edges to increase connectivity across bottlenecks. Techniques include stochastic edges, Personalized PageRank (PPR) edges, or curvature-based rewiring.

- **Use positional encodings or structural features**  
  Provide nodes with extra features that help distinguish them without sending all raw information through bottlenecks.

- **Higher-order or subgraph GNNs**  
  Capture more local structure, reducing the need for long-range message passing.

- **Residual mixing schemes (e.g., GCNII)**  
  Combine initial features with deep-layer outputs to preserve the original signal.

<hr>

## 7. Practical Advice

- **Assess task requirements**  
  Determine whether the task needs long-range or combinatorial information. For many standard semi-supervised node classification tasks on homophilous graphs, simple GNNs suffice.

- **Detect oversquashing**  
  For tasks connecting distant nodes across narrow cuts (e.g., molecular graphs or reasoning tasks), test by building synthetic bottleneck graphs or measuring sensitivity to added edges/virtual nodes.

- **Mitigate early**  
  Apply rewiring or virtual nodes as effective mitigation techniques before moving to heavier higher-order models.

**Summary:**  
Oversquashing is an information-theoretic bottleneck in message passing: when many messages are squeezed through a few edges or nodes, information is lost. Detecting it, analyzing it theoretically (via curvature or conductance), and mitigating it (via rewiring, residual connections, virtual nodes, or positional encodings) are active and practically important areas in modern GNN research.